# Experimento 006 - Catalogo e Compatibilidade de Modelo

Este notebook investiga a falha historica do Experimento 006: o backbone `fan_hybrid_tiny` nao existia com esse nome no registry/base experiments do TAO.

O notebook compara a configuracao historica com a configuracao corrigida `fan_small_12_p4_hybrid`, registra evidencias do catalogo e permite validar o treino corrigido.

## Perguntas

- `fan_hybrid_tiny` existe no catalogo TAO?
- Qual ID e nome do base experiment compativel?
- O backbone e compativel com `in_channels` do head?
- O dataset e compativel com a arquitetura?
- A configuracao corrigida pode executar treino real?
- Qual e a diferenca entre a configuracao historica e a corrigida?

## 1. Configuracao

Os dois controles GPU estao habilitados nesta execucao: primeiro reproduzimos a configuracao historica, depois validamos a corrigida. Isso criara dois jobs de treino separados.

In [1]:
import copy
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import requests

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
HISTORICAL_BACKBONE = "fan_hybrid_tiny"
CORRECTED_BACKBONE = "fan_small_12_p4_hybrid"
RUN_HISTORICAL_CONTROL = True
RUN_CORRECTED_TRAINING = True
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_006_EVIDENCE", "experiment-006-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
configured = os.getenv("TAO_SECRETS_FILE", "")
candidates = [Path(configured)] if configured else []
candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path.cwd().parent / "secrets.json", Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((item for item in candidates if item.exists()), None)
NGC_KEY = json.loads(secrets_file.read_text())["ngc_api_key"] if secrets_file else os.getenv("NGC_KEY", "")
if not NGC_KEY: raise RuntimeError("NGC key not found")
http = requests.Session()
print({"base_url": TAO_BASE_URL, "historical": HISTORICAL_BACKBONE, "corrected": CORRECTED_BACKBONE, "historical_control": RUN_HISTORICAL_CONTROL, "corrected_control": RUN_CORRECTED_TRAINING})

{'base_url': 'http://100.107.81.126:8090', 'historical': 'fan_hybrid_tiny', 'corrected': 'fan_small_12_p4_hybrid', 'historical_control': True, 'corrected_control': True}


## 2. Login e contexto TAO

In [2]:
def api(method, path, payload=None, headers=None):
    merged = {"Content-Type": "application/json"}
    if headers: merged.update(headers)
    response = http.request(method, TAO_BASE_URL + path, headers=merged, json=payload, timeout=(10, 120))
    response.raise_for_status()
    return response.json()

login = api("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
health = api("GET", "/api/v2/health")
workspace = api("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = api("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
schemas = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch=classification_pyt&action=train", headers=AUTH)
catalog = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch=classification_pyt", headers=AUTH)
assert "readiness" in health
assert workspace.get("id") == WORKSPACE_ID
print("context=PASS datasets=", len(datasets.get("datasets", [])), "schema=PASS")

context=PASS datasets= 1 schema=PASS


## 3. Analise do catalogo e compatibilidade

Esta celula nao cria jobs. Ela demonstra se o modelo historico esta registrado e compara o head com a configuracao corrigida.

In [3]:
experiments = catalog.get("experiments", [])
historical_matches = [item for item in experiments if HISTORICAL_BACKBONE.lower() in json.dumps(item).lower()]
base_matches = [item for item in experiments if item.get("id") == BASE_EXPERIMENT_ID]
default_spec = copy.deepcopy(schemas.get("default", {}))
historical = copy.deepcopy(default_spec)
historical.setdefault("model", {}).setdefault("backbone", {})["type"] = HISTORICAL_BACKBONE
corrected = copy.deepcopy(default_spec)
corrected.setdefault("model", {}).setdefault("backbone", {})["type"] = CORRECTED_BACKBONE
corrected.setdefault("model", {}).setdefault("head", {})["in_channels"] = 448
for spec in (historical, corrected):
    spec.setdefault("dataset", {}).update({"root_dir": DATASET_URI, "num_classes": 2})
    spec.setdefault("train", {}).update({"num_gpus": 1, "num_epochs": 1})
comparison = {"historical_matches": historical_matches, "base_matches": base_matches, "historical_backbone": HISTORICAL_BACKBONE, "corrected_backbone": CORRECTED_BACKBONE, "historical_spec": historical, "corrected_spec": corrected}
(EVIDENCE_DIR / "catalog-compatibility.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps({"historical_match_count": len(historical_matches), "base_experiment_found": bool(base_matches), "historical_backbone": HISTORICAL_BACKBONE, "corrected_backbone": CORRECTED_BACKBONE, "corrected_in_channels": corrected["model"]["head"].get("in_channels")}, indent=2))

{
  "historical_match_count": 1,
  "base_experiment_found": true,
  "historical_backbone": "fan_hybrid_tiny",
  "corrected_backbone": "fan_small_12_p4_hybrid",
  "corrected_in_channels": 448
}


## 4. Reproducao opcional da falha historica

Ative `RUN_HISTORICAL_CONTROL=True` somente para criar o job incompatível. O resultado esperado e uma falha terminal documentada.

In [4]:
def submit_and_wait(label, specs):
    payload = {"kind": "experiment", "name": f"experiment-006-{label}-{int(time.time())}", "network_arch": "classification_pyt", "encryption_key": "unittest-only", "workspace": WORKSPACE_ID, "action": "train", "specs": specs, "train_dataset_uris": [DATASET_URI], "eval_dataset_uri": DATASET_URI, "inference_dataset_uri": DATASET_URI, "base_experiment_ids": [BASE_EXPERIMENT_ID]}
    (EVIDENCE_DIR / f"{label}-payload.json").write_text(json.dumps(payload, indent=2))
    submitted = api("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
    job_id = submitted["id"]
    for _ in range(360):
        current = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{job_id}", headers=AUTH)
        (EVIDENCE_DIR / f"{label}-status.json").write_text(json.dumps(current, indent=2))
        state = current.get("status")
        print(label, state)
        if state in {"Done", "Completed", "Error", "Failed", "Failure", "Canceled", "Cancelled"}: return job_id, current
        time.sleep(10)
    raise TimeoutError(job_id)

historical_result = {"executed": False, "expected": "Error due to unregistered fan_hybrid_tiny"}
if RUN_HISTORICAL_CONTROL:
    historical_result = dict(zip(["job_id", "result"], submit_and_wait("historical", historical)))
    (EVIDENCE_DIR / "historical-result.json").write_text(json.dumps(historical_result, indent=2))
print(json.dumps(historical_result, indent=2))

historical Pending


historical Pending


historical Started


historical Started


historical Started


historical Started


historical Started


historical Started


historical Started


historical Error
{
  "job_id": "a1fcf6f6-16fb-432d-b02c-ec8094c30fd2",
  "result": {
    "id": "a1fcf6f6-16fb-432d-b02c-ec8094c30fd2",
    "created_on": "2026-10-19T20:31:02.674000+00:00",
    "last_modified": "2026-10-19T20:32:28.040000+00:00",
    "name": "experiment-006-historical-1792441799",
    "shared": false,
    "description": "",
    "version": "1.0.0",
    "logo": "https://www.nvidia.com",
    "ngc_path": "",
    "workspace": "98c4bec1-88df-423e-8a15-af78881e9742",
    "additional_id_info": null,
    "checkpoint_choose_method": "best_model",
    "checkpoint_epoch_number": {},
    "encryption_key": "unittest-only",
    "network_arch": "classification_pyt",
    "base_experiment_ids": [
      "469f42fe-37dc-55ff-b89b-5096477f9a87"
    ],
    "dataset_type": "image_classification",
    "dataset_formats": [
      "classification_pyt"
    ],
    "accepted_dataset_intents": [
      "training",
      "evaluation",
      "testing"
    ],
    "eval_dataset": null,
    "inference_datas

## 5. Treino corrigido opcional

Ative `RUN_CORRECTED_TRAINING=True` para validar a correcao com `fan_small_12_p4_hybrid`. O resultado esperado e `Done` com status detalhado `SUCCESS`, checkpoint e metricas.

In [5]:
corrected_result = {"executed": False, "expected": "Done with SUCCESS and metrics"}
if RUN_CORRECTED_TRAINING:
    corrected_result = dict(zip(["job_id", "result"], submit_and_wait("corrected", corrected)))
    (EVIDENCE_DIR / "corrected-result.json").write_text(json.dumps(corrected_result, indent=2))
print(json.dumps(corrected_result, indent=2))

corrected Pending


corrected Pending


corrected Started


corrected Started


corrected Started


corrected Started


corrected Started


corrected Started


corrected Started


corrected Done
{
  "job_id": "4c703a7e-1cd2-4152-aaf3-8288f6299eaf",
  "result": {
    "id": "4c703a7e-1cd2-4152-aaf3-8288f6299eaf",
    "created_on": "2026-10-19T20:32:34.483000+00:00",
    "last_modified": "2026-10-19T20:33:58.586000+00:00",
    "name": "experiment-006-corrected-1792441891",
    "shared": false,
    "description": "",
    "version": "1.0.0",
    "logo": "https://www.nvidia.com",
    "ngc_path": "",
    "workspace": "98c4bec1-88df-423e-8a15-af78881e9742",
    "additional_id_info": null,
    "checkpoint_choose_method": "best_model",
    "checkpoint_epoch_number": {
      "best_model_4c703a7e-1cd2-4152-aaf3-8288f6299eaf": 0,
      "latest_model_4c703a7e-1cd2-4152-aaf3-8288f6299eaf": 0
    },
    "encryption_key": "unittest-only",
    "network_arch": "classification_pyt",
    "base_experiment_ids": [
      "469f42fe-37dc-55ff-b89b-5096477f9a87"
    ],
    "dataset_type": "image_classification",
    "dataset_formats": [
      "classification_pyt"
    ],
    "accepted_data

## 6. Checklist final

`NOT_RUN` significa que o controle GPU estava desligado. A analise de catalogo e compatibilidade pode passar mesmo quando nenhum job foi criado.

In [6]:
checks = {
    "login_health_context": True,
    "historical_model_catalogued": bool(historical_matches),
    "base_experiment_found": bool(base_matches),
    "historical_context_documented": True,
    "corrected_context_documented": True,
    "historical_failure_reproduced": historical_result.get("result", {}).get("status") in {"Error", "Failed", "Failure"} if RUN_HISTORICAL_CONTROL else None,
    "corrected_training_passed": corrected_result.get("result", {}).get("status") in {"Done", "Completed"} if RUN_CORRECTED_TRAINING else None,
}
report = {"experiment": "006-training-real", "checks": checks, "generated_at": datetime.now(timezone.utc).isoformat(), "evidence_files": sorted(p.name for p in EVIDENCE_DIR.iterdir())}
(EVIDENCE_DIR / "final-checklist.json").write_text(json.dumps(report, indent=2))
for name, value in checks.items(): print(f"{name}: {value if value is not None else 'NOT_RUN'}")

login_health_context: True
historical_model_catalogued: True
base_experiment_found: True
historical_context_documented: True
corrected_context_documented: True
historical_failure_reproduced: NOT_RUN
corrected_training_passed: NOT_RUN
